In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
import warnings
warnings.filterwarnings("ignore")

source = r"C:\Users\X\Desktop\NNPR(RE)\yolov8\test\images\0a720df9-e4ef-4e44-8c13-b39b9be8444d___3e7fd381-0ae5-4421-8a70-279ee0ec1c61_Nissan-Terrano-6_jpg.rf.c2f78cb12de23cda1b1782f23ee505b0.jpg"
img = cv2.imread(source)
resized = cv2.resize(img, (640, 640))
device = "cuda"
model = YOLO(r"C:\Users\X\Desktop\X\NNPR\models\platedetectionv8.pt").to (device)

results = model.predict(resized, conf=0.7)

for result in results:
    boxes = result.boxes
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cropped = resized[y1:y2, x1:x2]
        gray_crop = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
        cv2.imshow(f"Cropped_{i}", gray_crop)
        cv2.rectangle(resized, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(resized, f"Plate {i+1}", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

cv2.imshow("Detections", resized)
cv2.waitKey(0)
cv2.destroyAllWindows()



0: 640x640 1 plate, 4.9ms
Speed: 2.9ms preprocess, 4.9ms inference, 53.8ms postprocess per image at shape (1, 3, 640, 640)


In [2]:
import cv2
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = YOLO(r"C:\Users\X\Desktop\X\NNPR\models\platedetectionv8.pt").to (device)

cap = cv2.VideoCapture("demo.mp4")

while cap.isOpened():
    ret, img = cap.read()
    if not ret:
        break

    img = cv2.resize(img, (640, 640))
    results = model.predict(img, conf=0.7, imgsz=640)

    for result in results:
        for i, box in enumerate(result.boxes):
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            crop = img[y1:y2, x1:x2]
            gray_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            cv2.imshow(f"Cropped Plate{i+1}", gray_crop)

            cv2.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(img, f"Plate {i+1}", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    cv2.imshow("Video Detection", img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 640x640 1 plate, 4.1ms
Speed: 2.1ms preprocess, 4.1ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 plate, 3.3ms
Speed: 2.2ms preprocess, 3.3ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 plate, 3.2ms
Speed: 1.7ms preprocess, 3.2ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 plate, 3.4ms
Speed: 1.8ms preprocess, 3.4ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 plate, 3.2ms
Speed: 1.9ms preprocess, 3.2ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 plate, 4.2ms
Speed: 1.7ms preprocess, 4.2ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 plate, 3.2ms
Speed: 2.0ms preprocess, 3.2ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 plate, 3.1ms
Speed: 1.7ms preprocess, 3.1ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 pl

In [3]:

import cv2
import torch
import easyocr
import numpy as np
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator, colors
import warnings
warnings.filterwarnings("ignore")

#YOLOV8+easyOCR

class ANPR:
    """Automatic Number Plate Recognition using Ultralytics YOLO and EasyOCR.

    This class handles license plate detection using a YOLO model and text extraction
    using EasyOCR. It supports both image and video streams for real-time inference.

    Attributes:
        model (YOLO): The YOLO model for license plate detection.
        reader (easyocr.Reader): The OCR reader instance for text recognition.
        device (torch.device): Computation device (CPU or CUDA).
    """

    def __init__(self, model_path):
        """Initializes the ANPR system."""
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = YOLO(model_path)
        self.reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())

    def detect_plates(self, im0):
        """Detects license plates in a image."""
        results = self.model.predict(im0)
        boxes = results[0].boxes.xyxy.cpu().numpy() if results and results[0].boxes is not None else []
        return boxes

    def extract_text(self, im0, bbox):
        """Performs OCR on the cropped license plate region."""
        x1, y1, x2, y2 = map(int, bbox)
        roi = im0[y1:y2, x1:x2]
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        text = self.reader.readtext(gray, detail=0, paragraph=True)
        return " ".join(text).strip() if text else ""

    def infer_video(self, source, output_path, display):
        """Performs real-time ANPR on a video stream."""
        cap = cv2.VideoCapture(source)
        if not cap.isOpened():
            raise ValueError(f"cannot open video source: {source}")

        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30

        writer = None
        if output_path:
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        while True:
            ret, im0 = cap.read()
            if not ret:
                break

            boxes = self.detect_plates(im0)
            ann = Annotator(im0, line_width=4)
            for bbox in boxes:
                text = self.extract_text(im0, bbox)
                ann.box_label(bbox, label=text, color=colors(17, True))

            if display:
                cv2.imshow("ANPR (press 'q' to exit)", im0)
            if writer:
                writer.write(im0)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

        cap.release()
        if writer:
            writer.release()
        cv2.destroyAllWindows()


if __name__ == "__main__":

    anpr = ANPR(model_path=r"C:\Users\X\Desktop\X\NNPR\models\platedetectionv8.pt")
    anpr.infer_video(source=r"C:\Users\X\Desktop\X\2023\demo.mp4", output_path="output.mp4", display=True)


0: 416x640 1 plate, 26.6ms
Speed: 1.7ms preprocess, 26.6ms inference, 1.2ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 plate, 3.2ms
Speed: 1.7ms preprocess, 3.2ms inference, 1.5ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 plate, 4.3ms
Speed: 1.4ms preprocess, 4.3ms inference, 1.2ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 plate, 3.5ms
Speed: 1.2ms preprocess, 3.5ms inference, 1.0ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 2 plates, 3.1ms
Speed: 1.4ms preprocess, 3.1ms inference, 1.2ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 2 plates, 4.2ms
Speed: 1.5ms preprocess, 4.2ms inference, 1.4ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 plate, 4.2ms
Speed: 1.4ms preprocess, 4.2ms inference, 1.7ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 plate, 3.5ms
Speed: 1.2ms preprocess, 3.5ms inference, 1.2ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 